# MV-SAM3D Stage-2 LoRA — micro-batch overfitting PoC

**Goal:** inject PEFT LoRA into the Stage-2 SLAT Flow Transformer and show flow-matching loss converges on a `va_split` micro-batch.

**Install:** same self-contained sequence as `sam_baneus_ultrra.ipynb` / `mvsam3d_example_current.ipynb` (torch 2.5.1, compiled pkgs backup **or** §4d build, Hydra patch).

### Before you start
1. **A100 80GB**
2. HF access to gated [`facebook/sam-3d-objects`](https://huggingface.co/facebook/sam-3d-objects) + Colab secret `HF_TOKEN`
3. Ideal: `/content/drive/MyDrive/colab_compiled_pkgs.tar.gz` from a prior `mvsam3d_example_current` backup (skips a long §4d). If missing, this notebook builds those wheels.

**Restart gotcha:** Step 0 may pin `torch==2.5.1+cu121` and ask you to **Runtime → Restart session**, then continue from the clone cell (do **not** re-pin torch after restart).


## 0 — Drive + GPU + pin torch 2.5.1 (Step 0)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import torch, subprocess, sys, os

assert torch.cuda.is_available(), "No GPU — enable A100 runtime first!"
print(f"GPU:     {torch.cuda.get_device_name(0)}")
print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")

TARGET_TORCH = "2.5.1"
if not torch.__version__.startswith(TARGET_TORCH):
    print(f"\n▶ Downgrading torch {torch.__version__} → {TARGET_TORCH}+cu121 …")
    !pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
    !pip install -q xformers==0.0.28.post3
    print("✓ Torch stack pinned. ⚠ Restart the runtime now!")
    print("   Runtime → Restart session, then continue from the NEXT markdown cell (clone).")
else:
    print(f"\n✓ torch {torch.__version__} is the target version")


> **If Step 0 printed "Restart the runtime now!":** Restart session, then continue **from this cell** (clone). Do **not** re-run Step 0 after restart.


## 1 — Clone repos + numpy/scipy pin


In [ ]:
# Same remotes / pins as sam_baneus_ultrra.ipynb
from pathlib import Path
import os, sys

ROOT         = Path("/content")
SAM3D_ROOT   = ROOT / "sam-3d-objects"
DA3_ROOT     = ROOT / "Depth-Anything-3"
MVSAM3D_ROOT = ROOT / "MV-SAM3D"

_np_pin = ROOT / "_numpy_pin.txt"
_np_pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_np_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"])

!pip install -q "numpy==1.26.4" "scipy==1.13.1"
import importlib, scipy, numpy
importlib.reload(scipy)
print(f"✓ numpy {numpy.__version__}  scipy {scipy.__version__}")

%cd /content
REPOS = {
    "sam-3d-objects":   "https://github.com/facebookresearch/sam-3d-objects.git",
    "Depth-Anything-3": "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
    # same remote as sam_baneus_ultrra (has notebook/inference.py)
    "MV-SAM3D":         "https://github.com/devinli123/MV-SAM3D.git",
}
for name, url in REPOS.items():
    if os.path.isdir(name):
        print(f"OK   {name}/ already exists")
    else:
        !git clone --depth 1 {url}
        print(f"cloned {name}")

_inf = MVSAM3D_ROOT / "notebook" / "inference.py"
print("inference.py:", _inf.exists(), _inf)
if not _inf.exists():
    # fallback: link from sam-3d-objects
    (MVSAM3D_ROOT / "notebook").mkdir(parents=True, exist_ok=True)
    for fn in ("inference.py", "load_images_and_masks.py", "mesh_alignment.py"):
        src = SAM3D_ROOT / "notebook" / fn
        dst = MVSAM3D_ROOT / "notebook" / fn
        if src.exists() and not dst.exists():
            os.symlink(src, dst)
            print("linked", dst)
print("inference.py now:", (MVSAM3D_ROOT / "notebook" / "inference.py").exists())


## 1b — DA3 + core SAM3D deps (same as example §3–§4c)


In [ ]:
!pip install -q xformers
!pip install -q -e /content/Depth-Anything-3
print("✓ Depth Anything 3 installed")


In [ ]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]
print(f"Targeting torch {TORCH_VERSION}  CUDA {torch.version.cuda}")

!pip install -q --no-deps -e /content/sam-3d-objects

!pip install -q \
    hydra-core==1.3.2 \
    omegaconf \
    loguru \
    opencv-python-headless \
    trimesh \
    roma \
    seaborn \
    safetensors \
    tqdm \
    einops \
    astor \
    lightning \
    xatlas \
    pyvista \
    pymeshfix \
    igraph \
    "numpy<2" \
    "huggingface_hub[cli]" \
    gradio \
    peft==0.13.2 \
    accelerate \
    matplotlib \
    pillow

!pip install -q open3d
!pip install -q --no-deps "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900"
!pip install -q --no-deps "MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b"

print("✓ Core SAM 3D deps + peft installed")


## 1c — Compiled CUDA pkgs (restore backup **or** build §4d)

Same as `mvsam3d_example_current`: prefer Drive `colab_compiled_pkgs.tar.gz`; otherwise install pytorch3d / flash-attn / kaolin / gsplat / spconv.


In [ ]:
import os, site, sys, subprocess, torch
from pathlib import Path

sp = site.getsitepackages()[0]
BACKUP_CANDIDATES = [
    Path("/content/drive/MyDrive/colab_compiled_pkgs.tar.gz"),
    Path("/content/drive/MyDrive/SatelliteFuse/colab_compiled_pkgs.tar.gz"),
]
backup = next((p for p in BACKUP_CANDIDATES if p.exists()), None)

def _compiled_ok():
    try:
        import pytorch3d, flash_attn, kaolin, gsplat, spconv  # noqa: F401
        return True
    except Exception as e:
        print("compiled import check:", e)
        return False

if backup is not None:
    print("Restoring compiled pkgs from", backup)
    !tar xzf {backup} -C {sp}
    !pip install -q numpy==1.26.4 scipy==1.13.1
    if _compiled_ok():
        print("✓ All compiled packages restored")
    else:
        print("⚠ Backup restored but imports failed — falling through to §4d build")
        backup = None

if backup is None:
    print("No usable backup — running §4d install (can take a while)…")
    TORCH_VER = torch.__version__.split("+")[0]
    CUDA_VER  = "cu" + torch.version.cuda.replace(".", "")
    PY_VER    = f"{sys.version_info.major}{sys.version_info.minor}"
    print(f"torch {TORCH_VER}  CUDA {torch.version.cuda}  Python {PY_VER}")
    assert TORCH_VER.startswith("2.5"), (
        f"Expected torch 2.5.x but got {TORCH_VER}. Re-run Step 0 + restart."
    )

    from pathlib import Path as _Path
    _pin = _Path("/content/_numpy_pin.txt")
    _pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
    os.environ["PIP_CONSTRAINT"] = str(_pin)

    !pip install -q ninja packaging

    # PyTorch3D
    try:
        import pytorch3d
        print(f"✓ pytorch3d {pytorch3d.__version__}")
    except ImportError:
        print("Installing PyTorch3D …")
        ret = subprocess.run(
            f"pip install --no-index --no-cache-dir pytorch3d "
            f"-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/"
            f"py{PY_VER}_{CUDA_VER}_pyt{TORCH_VER.replace('.','')}/download.html",
            shell=True,
        )
        if ret.returncode != 0:
            !pip install "pytorch3d @ git+https://github.com/facebookresearch/pytorch3d.git"
        import pytorch3d
        print(f"✓ pytorch3d {pytorch3d.__version__}")

    # flash-attn
    try:
        import flash_attn
        print(f"✓ flash_attn {flash_attn.__version__}")
    except ImportError:
        print("Installing flash-attn …")
        !pip install -q flash-attn --no-build-isolation
        import flash_attn
        print(f"✓ flash_attn {flash_attn.__version__}")

    # kaolin
    os.environ["PIP_FIND_LINKS"] = (
        f"https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-{TORCH_VER}_{CUDA_VER}.html"
    )
    try:
        import kaolin
        print(f"✓ kaolin {kaolin.__version__}")
    except ImportError:
        print("Installing kaolin …")
        !pip install -q kaolin
        import kaolin
        print(f"✓ kaolin {kaolin.__version__}")

    # gsplat
    try:
        import gsplat
        print(f"✓ gsplat {getattr(gsplat, '__version__', 'ok')}")
    except ImportError:
        print("Installing gsplat …")
        !pip install -q --no-build-isolation \
            "gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70"
        import gsplat
        print("✓ gsplat installed")

    # spconv
    try:
        import spconv
        print("✓ spconv installed")
    except Exception:
        print("Installing spconv …")
        cumm_inc = os.path.join(sp, "cumm", "include")
        os.environ["CPLUS_INCLUDE_PATH"] = cumm_inc + ":" + os.environ.get("CPLUS_INCLUDE_PATH", "")
        os.environ["C_INCLUDE_PATH"] = cumm_inc + ":" + os.environ.get("C_INCLUDE_PATH", "")
        !pip install -q --force-reinstall cumm-cu121
        !pip install -q --force-reinstall spconv-cu121
        try:
            import spconv
            print("✓ spconv installed")
        except Exception as e:
            os.environ["CUMM_DISABLE_JIT"] = "1"
            import spconv
            print("✓ spconv installed (JIT disabled)", e)

    # optional: write backup for next time
    out_backup = Path("/content/drive/MyDrive/colab_compiled_pkgs.tar.gz")
    try:
        !tar czf {out_backup} -C {sp} pytorch3d flash_attn kaolin gsplat spconv cumm \
            $(cd {sp} && find . -maxdepth 1 -name "flash_attn_2_cuda*" -printf "%f\n") \
            $(cd {sp} && find . -maxdepth 1 -name "*.dist-info" \( -name "pytorch3d*" -o -name "flash_attn*" -o -name "kaolin*" -o -name "gsplat*" -o -name "spconv*" -o -name "cumm*" \) -printf "%f\n")
        print(f"✓ Saved backup to {out_backup} ({out_backup.stat().st_size/1e6:.0f} MB)")
    except Exception as e:
        print("backup save skipped:", e)

print("✓ compiled stack ready")


## 1d — Hydra patch + CUDA env (same as example §4e)


In [ ]:
import os
cuda_home = "/usr/local/cuda"
os.environ["CUDA_HOME"] = cuda_home
os.environ["CONDA_PREFIX"] = cuda_home
os.environ["LIDRA_SKIP_INIT"] = "true"
!python /content/sam-3d-objects/patching/hydra
print("✓ Hydra patched")


## 2 — HF login + SAM 3D checkpoints


In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone
from huggingface_hub import login, snapshot_download

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
assert hf_token, "Set Colab secret HF_TOKEN after requesting facebook/sam-3d-objects access"
login(token=hf_token)

SAM3D_ROOT = Path("/content/sam-3d-objects")
MVSAM3D_ROOT = Path("/content/MV-SAM3D")
ckpt_hf = SAM3D_ROOT / "checkpoints" / "hf"

need_dl = not (
    (ckpt_hf / "pipeline.yaml").exists()
    or (ckpt_hf / "checkpoints" / "pipeline.yaml").exists()
)
if need_dl:
    print("Downloading facebook/sam-3d-objects …")
    snapshot_download(repo_id="facebook/sam-3d-objects", token=hf_token, local_dir=str(ckpt_hf))
else:
    print("✓ checkpoint present under", ckpt_hf)

dst = MVSAM3D_ROOT / "checkpoints" / "hf"
src_nested = ckpt_hf / "checkpoints"
src = src_nested if (src_nested / "pipeline.yaml").exists() else ckpt_hf
if not dst.exists():
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        os.symlink(src, dst)
        print("symlinked", dst, "→", src)

PIPELINE_YAML = dst / "pipeline.yaml"
assert PIPELINE_YAML.exists(), PIPELINE_YAML
print("PIPELINE_YAML =", PIPELINE_YAML)

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
DRIVE_OUT = Path("/content/drive/MyDrive/SatelliteFuse/Stage2LoRA") / RUN_TS / "va_split"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("DRIVE_OUT =", DRIVE_OUT)


## 3 — Hyperparameters


In [ ]:
from pathlib import Path

CASE_NAME = "va_split"
DATA_ROOT = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")
AERIAL_DIR = DATA_ROOT / "satellite"
IMG_SUBDIR, MASK_SUBDIR = "images", "masks"

MICRO_N = 8
SEED = 0
STAGE1_STEPS = 25
STAGE2_TEACHER_STEPS = 25
TRAIN_STEPS = 200
LR = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGETS = ["to_qkv", "to_q", "to_kv", "to_out"]
LOG_EVERY = 10

OUTPUT_DIR = Path("/content/stage2_lora_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MICRO_N={MICRO_N} | LoRA r={LORA_R} α={LORA_ALPHA}")
print(f"targets={LORA_TARGETS}")
print(f"DATA_ROOT exists={DATA_ROOT.exists()} | TRAIN_STEPS={TRAIN_STEPS}")


## 4 — Load Stage-2 pipeline (no LoRA yet)

Load checkpoints first so the **teacher** Stage-2 sample is from the frozen base model.
LoRA is injected in §6 *after* caching `x1`.


In [ ]:
import sys, importlib.util, torch
from pathlib import Path

_candidates = [
    Path("/content/MV-SAM3D"),
    Path("/content/drive/MyDrive/SatelliteFuse/code/MV-SAM3D"),
    Path("/content/sam-3d-objects"),
]
MVSAM3D_ROOT = None
INF_PY = None
for root in _candidates:
    cand = root / "notebook" / "inference.py"
    if cand.exists():
        MVSAM3D_ROOT, INF_PY = root, cand
        break

if INF_PY is None:
    raise FileNotFoundError(
        "notebook/inference.py not found.\n"
        "Re-run §1, or copy your local MV-SAM3D (with notebook/) to Colab.\n"
        f"Checked: {[str(c) for c in _candidates]}"
    )

print("MVSAM3D_ROOT =", MVSAM3D_ROOT)
print("inference.py =", INF_PY)

nb_dir = str(INF_PY.parent)
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)
if str(MVSAM3D_ROOT) not in sys.path:
    sys.path.insert(0, str(MVSAM3D_ROOT))

try:
    from inference import Inference
except ModuleNotFoundError:
    spec = importlib.util.spec_from_file_location("mvsam3d_notebook_inference", INF_PY)
    mod = importlib.util.module_from_spec(spec)
    sys.modules["mvsam3d_notebook_inference"] = mod
    spec.loader.exec_module(mod)
    Inference = mod.Inference
    print("Loaded Inference via importlib from", INF_PY)

PIPELINE_YAML = Path("/content/MV-SAM3D/checkpoints/hf/pipeline.yaml")
if not PIPELINE_YAML.exists():
    PIPELINE_YAML = MVSAM3D_ROOT / "checkpoints" / "hf" / "pipeline.yaml"
assert PIPELINE_YAML.exists(), f"Missing {PIPELINE_YAML} — re-run §2 checkpoint symlink"

print("Loading Inference pipeline…")
inference = Inference(str(PIPELINE_YAML), compile=False)
pipeline = inference._pipeline
device = torch.device("cuda")

slat_gen = pipeline.models["slat_generator"]
cfg_mod = slat_gen.reverse_fn
backbone = cfg_mod.backbone
print("slat_generator:", type(slat_gen).__name__)
print("reverse_fn:", type(cfg_mod).__name__)
print("backbone:", type(backbone).__name__)

n_params = sum(p.numel() for p in slat_gen.parameters())
print(f"Stage-2 parameter count (base): {n_params:,}")


## 5 — Micro-batch + Stage-1 coords + teacher `x1`

Teacher sample uses the **base** Stage-2 weights (LoRA not injected yet).


In [ ]:
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from sam3d_objects.pipeline.multi_view_utils import inject_generator_multi_view

img_dir = AERIAL_DIR / IMG_SUBDIR
mask_dir = AERIAL_DIR / MASK_SUBDIR
assert img_dir.is_dir(), img_dir

def _natkey(p):
    s = Path(p).stem
    return (0, int(s), s) if s.isdigit() else (1, 0, s)

img_paths = sorted(
    [p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}],
    key=_natkey,
)[:MICRO_N]
assert len(img_paths) == MICRO_N, f"need {MICRO_N} images, got {len(img_paths)}"
print("stems:", [p.stem for p in img_paths])

view_images, view_masks = [], []
for ip in img_paths:
    view_images.append(Image.open(ip).convert("RGB"))
    mp = mask_dir / f"{ip.stem}.png"
    if not mp.exists():
        cands = list(mask_dir.glob(ip.stem + "*"))
        assert cands, f"no mask for {ip}"
        mp = cands[0]
    view_masks.append(Image.open(mp))


def _as_alpha(mask) -> np.ndarray:
    """Force mask to HxWx1 uint8 in {0,255}. ULTRRA masks are often RGBA/L."""
    m = np.asarray(mask)
    if m.ndim == 3:
        # RGBA → alpha; RGB → max channel; otherwise first channel
        m = m[..., 3] if m.shape[-1] == 4 else m[..., 0]
    if m.dtype == bool:
        m = m.astype(np.uint8) * 255
    elif m.dtype != np.uint8:
        m = (m * 255).astype(np.uint8) if float(m.max()) <= 1.0 else m.astype(np.uint8)
    return m[..., None]  # HxWx1


def preprocess_views(pipeline, view_images, view_masks=None, view_pointmaps=None):
    n = len(view_images)
    if view_masks is None:
        view_masks = [None] * n
    if view_pointmaps is None:
        view_pointmaps = [None] * n
    ss_dicts, slat_dicts = [], []
    for image, mask, ext_pm in zip(view_images, view_masks, view_pointmaps):
        image = np.asarray(image)
        if image.ndim == 2:
            image = np.stack([image] * 3, axis=-1)
        image = image[..., :3]
        if mask is not None:
            alpha = _as_alpha(mask)
            rgba = np.concatenate([image.astype(np.uint8), alpha], axis=-1)
        else:
            alpha = np.full((*image.shape[:2], 1), 255, np.uint8)
            rgba = np.concatenate([image.astype(np.uint8), alpha], axis=-1)
        assert rgba.shape[-1] == 4, rgba.shape
        rgba_pil = Image.fromarray(rgba, mode="RGBA")
        if hasattr(pipeline, "compute_pointmap"):
            if ext_pm is not None:
                pmd = pipeline.compute_pointmap(rgba_pil, pointmap=torch.from_numpy(ext_pm).float())
            else:
                pmd = pipeline.compute_pointmap(rgba_pil, pointmap=None)
            pointmap = pmd["pointmap"]
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor, pointmap=pointmap))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))
        else:
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))
    return ss_dicts, slat_dicts


torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Preprocessing {MICRO_N} views…")
ss_dicts, slat_dicts = preprocess_views(pipeline, view_images, view_masks)

print("Stage-1 multi-view → coords…")
with torch.no_grad():
    ss_ret = pipeline.sample_sparse_structure_multi_view(
        ss_dicts, inference_steps=STAGE1_STEPS, mode="multidiffusion",
    )
coords = ss_ret["coords"]
print("coords:", tuple(coords.shape))

print("Teacher Stage-2 (base weights) → x1…")
slat_gen.no_shortcut = True
cfg_mod.strength = float(getattr(pipeline, "slat_cfg_strength", 0.0) or 0.0)
with torch.no_grad():
    with torch.autocast(device_type="cuda", dtype=pipeline.dtype):
        cond_args, cond_kwargs = pipeline.get_multi_view_condition_input(
            pipeline.condition_embedders["slat_condition_embedder"],
            slat_dicts,
            pipeline.slat_condition_input_mapping,
        )
        cond_args = tuple(cond_args) + (coords.detach().cpu().numpy(),)
        latent_shape = (slat_dicts[0]["image"].shape[0], coords.shape[0], 8)
        prev_steps = slat_gen.inference_steps
        slat_gen.inference_steps = STAGE2_TEACHER_STEPS
        with inject_generator_multi_view(
            slat_gen,
            num_views=len(slat_dicts),
            num_steps=slat_gen.inference_steps,
            mode="multidiffusion",
        ):
            teacher = slat_gen(latent_shape, device, *cond_args, **cond_kwargs)
        slat_gen.inference_steps = prev_steps
        x1 = teacher[0].detach().float()
        if x1.dim() == 2:
            x1 = x1.unsqueeze(0)

cond_args_train = []
for a in cond_args[:-1]:
    cond_args_train.append(a.detach().to(device) if torch.is_tensor(a) else a)
cond_args_train.append(cond_args[-1])
cond_args_train = tuple(cond_args_train)
cond_kwargs_train = {k: (v.detach() if torch.is_tensor(v) else v) for k, v in cond_kwargs.items()}
x1 = x1.to(device)

# FlowMatching.loss calls reverse_fn once — multi-view injection only patches the
# ODE sampler (_generate_dynamics), so stacked (N_views, …) conds crash cross-attn
# (q batch=1 vs kv batch=8). Train with view-0 tokens (same pose convention as MV fused pose).
_cond0 = cond_args_train[0]
if torch.is_tensor(_cond0) and _cond0.shape[0] == len(slat_dicts):
    _cond_single = _cond0[0]  # drop view axis → (B, T, D) or (T, D)
    if _cond_single.dim() == 2:
        _cond_single = _cond_single.unsqueeze(0)
    cond_args_fm = (_cond_single,) + cond_args_train[1:]
    print(f"train cond: multi-view {_cond0.shape} → view0 {_cond_single.shape}")
else:
    cond_args_fm = cond_args_train
    print("train cond: using as-is", type(_cond0), getattr(_cond0, "shape", None))

print(f"x1 shape={tuple(x1.shape)}")


## 6 — Inject LoRA into Stage-2 backbone

Freeze base weights; only LoRA adapters are trainable.


In [ ]:
from peft import LoraConfig, get_peft_model

for p in slat_gen.parameters():
    p.requires_grad = False

hits = [n for n, _ in backbone.named_modules() if n.split(".")[-1] in set(LORA_TARGETS)]
print(f"LoRA name hits: {len(hits)}  e.g. {hits[:8]}")
assert hits, "No modules match LORA_TARGETS"

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0.0,
    bias="none",
)
peft_backbone = get_peft_model(backbone, lora_cfg)
cfg_mod.backbone = peft_backbone

trainable = sum(p.numel() for p in slat_gen.parameters() if p.requires_grad)
total = sum(p.numel() for p in slat_gen.parameters())
print(f"slat_generator trainable: {trainable:,} / {total:,}  ({100.0 * trainable / max(total,1):.4f}%)")
peft_backbone.print_trainable_parameters()

cfg_mod.strength = 0.0
slat_gen.no_shortcut = True
print("CFG strength = 0 for FM training")


## 7 — Flow-matching train loop + loss curve

Calls library `FlowMatching.loss` (velocity MSE). Saves curve + adapter to Drive.


In [ ]:
import json, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from sam3d_objects.model.backbone.generator.flow_matching.model import FlowMatching
from sam3d_objects.pipeline.multi_view_utils import inject_generator_multi_view

opt = torch.optim.AdamW(
    [p for p in slat_gen.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

history = []
slat_gen.train()
peft_backbone.train()

print(f"Training {TRAIN_STEPS} steps | lr={LR}")
for step in range(TRAIN_STEPS):
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=pipeline.dtype):
        # Single-view cond (view 0). Do NOT use inject_generator_multi_view here —
        # it only wraps _generate_dynamics (sampling), not reverse_fn (FM loss).
        loss, _details = FlowMatching.loss(
            slat_gen, x1, *cond_args_fm, **cond_kwargs_train
        )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in slat_gen.parameters() if p.requires_grad], 1.0
    )
    opt.step()

    lv = float(loss.detach().float().cpu())
    history.append(lv)
    if step % LOG_EVERY == 0 or step == TRAIN_STEPS - 1:
        print(f"step {step:4d}/{TRAIN_STEPS}  flow_matching_loss={lv:.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history, lw=1.5)
ax.set_xlabel("step")
ax.set_ylabel("flow-matching loss")
ax.set_title(f"Stage-2 LoRA overfit (va_split N={MICRO_N}, r={LORA_R})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
curve_path = OUTPUT_DIR / "stage2_lora_loss_curve.png"
fig.savefig(curve_path, dpi=140, bbox_inches="tight")
plt.show()
print("Saved", curve_path)

adapter_dir = OUTPUT_DIR / "lora_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
peft_backbone.save_pretrained(str(adapter_dir))
np.savez(OUTPUT_DIR / "loss_history.npz", loss=np.asarray(history, np.float64))
meta = {
    "MICRO_N": MICRO_N,
    "TRAIN_STEPS": TRAIN_STEPS,
    "LR": LR,
    "LORA_R": LORA_R,
    "LORA_ALPHA": LORA_ALPHA,
    "LORA_TARGETS": LORA_TARGETS,
    "loss_first": history[0],
    "loss_last": history[-1],
    "stems": [p.stem for p in img_paths],
    "trainable_params": int(trainable),
    "total_params": int(total),
}
(OUTPUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2))
print("loss first→last:", round(history[0], 6), "→", round(history[-1], 6))

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for p in [curve_path, OUTPUT_DIR / "loss_history.npz", OUTPUT_DIR / "run_meta.json"]:
    shutil.copy2(p, DRIVE_OUT / p.name)
_dest = DRIVE_OUT / "lora_adapter"
if _dest.exists():
    shutil.rmtree(_dest)
shutil.copytree(adapter_dir, _dest)
print("→ Drive", DRIVE_OUT)


## 8 — Success criteria / what to send back

- Trainable ≪ total params
- Loss curve trends down
- Drive: `SatelliteFuse/Stage2LoRA/<timestamp>/va_split/stage2_lora_loss_curve.png`

Paste the §6 trainable print + the curve (or `run_meta.json` first/last loss).
